# RY3 EVAL REPAIR — fixed judge harness

## What broke last time
The v02 evaluation judge was supposed to emit strict JSON scores, but instead it usually emitted free prose (it answered the eval questions rather than grading them). The parser then failed on every single output — 240/240 `parse_error` — and **silently converted every failure into zeros**. Those zeros were recorded as if they were real model scores, invalidating the whole v02 comparison (`RY3_v02_MANIFEST.json` flags `judge_scores_are_not_valid: true`).

## What this notebook does differently
1. **Prompt demands ONLY JSON** — the system prompt instructs the judge to output a single JSON object: no prose, no markdown fences.
2. **Parser never silently zeros** — strict `json.loads` first, then a `{...}`-extraction fallback; every output is validated (all 6 keys present, integer scores 0–4, non-empty reason). Failures are recorded as `parsed: false` **with the raw output and an error reason** — never as fake zeros.
3. **TEST_MODE gate** — runs 10 examples first and prints each raw output + parse result, so you verify the judge behaves before committing to 720 generations.
4. **Incremental writes** — judgments are appended to disk as they complete, so a crash or Colab timeout loses nothing; re-running resumes where it left off.
5. **Same locked 240 evals × 3 models** (clean base, v01, v02) = 720 judgments, then a per-model summary and a per-eval-id comparison table.

Judge model: `Qwen/Qwen3-1.7B` — the **clean base model**, not the heretic variant (per project convention).


## 1. Install dependencies

In [ ]:
!pip install -U transformers accelerate --quiet
import json, re, os, random, collections
print("imports ok")

## 2. Upload the three results files
Upload these three files when prompted (rename on your machine first if needed):
- `base_results.jsonl` — 240 rows, keys: id, instruction, expected, response
- `ry3_results.jsonl` — 240 rows (v01), keys: id, instruction, expected, response
- `v02_results.jsonl` — 240 rows, keys: id, instruction, expected_output, ry3_v02_output

In [ ]:
from google.colab import files
uploaded = files.upload()
print("uploaded:", list(uploaded.keys()))

## 3. Load + normalize to a uniform schema

In [ ]:
def load_uniform(path, model_name, keys):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            d = json.loads(line)
            rows.append({
                "eval_id": d[keys["id"]],
                "model": model_name,
                "instruction": d[keys["instruction"]],
                "expected": d[keys["expected"]],
                "response": d[keys["response"]],
            })
    return rows

base = load_uniform("base_results.jsonl", "base",
                    {"id": "id", "instruction": "instruction", "expected": "expected", "response": "response"})
v01 = load_uniform("ry3_results.jsonl", "v01",
                   {"id": "id", "instruction": "instruction", "expected": "expected", "response": "response"})
v02 = load_uniform("v02_results.jsonl", "v02",
                   {"id": "id", "instruction": "instruction", "expected": "expected_output", "response": "ry3_v02_output"})

assert len(base) == 240, f"base has {len(base)} rows, expected 240"
assert len(v01) == 240, f"v01 has {len(v01)} rows, expected 240"
assert len(v02) == 240, f"v02 has {len(v02)} rows, expected 240"

items = base + v01 + v02
assert len(items) == 720

# sanity: the same 240 eval_ids must appear for all three models
ids_base = {r["eval_id"] for r in base}
assert ids_base == {r["eval_id"] for r in v01} == {r["eval_id"] for r in v02}, "eval_id sets differ across models!"
print(f"OK — base={len(base)} v01={len(v01)} v02={len(v02)} total={len(items)}; eval_ids align across models")

## 4. Load the judge model (clean Qwen3-1.7B base — NOT heretic)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

JUDGE_MODEL = "Qwen/Qwen3-1.7B"   # clean base model, per project convention
tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL,
    torch_dtype=torch.bfloat16,   # ~3.4 GB in bf16 — fits a Colab T4 comfortably
    device_map="auto",
)
model.eval()
print("judge loaded:", JUDGE_MODEL)

## 5. Judge prompt — demands ONLY a single JSON object

In [ ]:
JUDGE_SYSTEM = """You are an impartial evaluator. Your job is to grade an AI assistant's response to a user instruction.
You MUST output ONLY a single JSON object with exactly these keys:
- "correctness": integer 0-4
- "relevance": integer 0-4
- "completeness": integer 0-4
- "instruction_following": integer 0-4
- "reasoning_quality": integer 0-4
- "brief_reason": one short sentence explaining the scores
Rubric for each 0-4 dimension: 0 = total failure, 1 = poor, 2 = acceptable, 3 = good, 4 = excellent.
STRICT RULES: Output ONLY the JSON object. No prose before or after it. No markdown code fences. No explanations outside the JSON. Start your output with { and end it with }."""

def build_judge_messages(item):
    user = (
        "INSTRUCTION GIVEN TO THE ASSISTANT:\n" + item["instruction"] +
        "\n\nREFERENCE (EXPECTED) ANSWER:\n" + item["expected"] +
        "\n\nASSISTANT'S ACTUAL RESPONSE:\n" + item["response"] +
        "\n\nGrade the assistant's response on the five dimensions. Output ONLY the JSON object."
    )
    return [
        {"role": "system", "content": JUDGE_SYSTEM},
        {"role": "user", "content": user},
    ]

def render_prompt(messages):
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        # base model without a chat template: fall back to a plain labeled format
        parts = [f"{m['role'].upper()}:\n{m['content']}" for m in messages]
        return "\n\n".join(parts) + "\n\nASSISTANT:\n"

# preview the rendered prompt on the first item
print(render_prompt(build_judge_messages(items[0]))[:1500])
# IMPORTANT: Qwen3-base emits <think>...</think> reasoning that exhausts the token budget.
# The system prompt below explicitly forbids think tags.


## 6. Generation — low temperature, enough tokens for the JSON

In [ ]:
def judge_generate(messages, max_new_tokens=1024, temperature=0.1):
    prompt = render_prompt(messages)
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen_ids = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

## 7. Robust parser — NEVER silently converts a failure into zeros

In [ ]:
REQUIRED_KEYS = ["correctness", "relevance", "completeness",
                 "instruction_following", "reasoning_quality", "brief_reason"]
SCORE_KEYS = REQUIRED_KEYS[:5]

import re as _re
_THINK_RE = _re.compile(r"<think>.*?</think>", _re.DOTALL)

def _strip_think(text):
    """Remove Qwen3 <think> reasoning blocks before JSON extraction."""
    return _THINK_RE.sub("", text).strip()

def parse_judge_output(raw, eval_id, model_name):
    raw = _strip_think(raw)
    rec = {"eval_id": eval_id, "model": model_name}

    def fail(reason):
        # A parser failure is recorded as parsed=False WITH the raw output.
        # It is NEVER turned into fake zero scores.
        rec.update({"parsed": False, "parse_error": reason, "raw_judge_output": raw})
        return rec

    data = None
    try:
        data = json.loads(raw)                    # attempt 1: strict parse
    except Exception:
        m = re.search(r"\{.*\}", raw, re.DOTALL)   # attempt 2: first {...} block
        if not m:
            return fail("no JSON object found in judge output")
        try:
            data = json.loads(m.group(0))
        except Exception as e:
            return fail(f"JSON decode failed after brace extraction: {e}")

    if not isinstance(data, dict):
        return fail("parsed JSON is not an object")
    missing = [k for k in REQUIRED_KEYS if k not in data]
    if missing:
        return fail(f"missing required keys: {missing}")
    for k in SCORE_KEYS:
        v = data[k]
        if isinstance(v, bool) or not isinstance(v, int):
            return fail(f"score '{k}' is not an integer: {v!r}")
        if not (0 <= v <= 4):
            return fail(f"score '{k}' out of range 0-4: {v}")
    brief = data["brief_reason"]
    if not isinstance(brief, str) or not brief.strip():
        return fail("brief_reason is not a non-empty string")

    rec.update({
        "parsed": True,
        "correctness": data["correctness"],
        "relevance": data["relevance"],
        "completeness": data["completeness"],
        "instruction_following": data["instruction_following"],
        "reasoning_quality": data["reasoning_quality"],
        "brief_reason": brief.strip(),
        "raw_judge_output": raw,
    })
    return rec

def run_judge(item):
    raw = judge_generate(build_judge_messages(item))
    return parse_judge_output(raw, item["eval_id"], item["model"])

# parser unit tests (no model needed)
_good = '{"correctness": 3, "relevance": 4, "completeness": 3, "instruction_following": 4, "reasoning_quality": 2, "brief_reason": "Clear and on-topic."}'
assert parse_judge_output(_good, "x", "base")["parsed"] is True
_bad = parse_judge_output("The response is pretty good overall.", "x", "base")
assert _bad["parsed"] is False and "raw_judge_output" in _bad and "parse_error" in _bad
_oob = parse_judge_output('{"correctness": 9, "relevance": 4, "completeness": 3, "instruction_following": 4, "reasoning_quality": 2, "brief_reason": "x"}', "x", "base")
assert _oob["parsed"] is False
print("parser unit tests passed")

## 8. TEST MODE — run this first, verify the judge, then flip the flag

In [ ]:
TEST_MODE = True   # <-- set to False after the test below looks clean

if TEST_MODE:
    rng = random.Random(3407)
    sample = []
    for mname, n in (("base", 4), ("v01", 3), ("v02", 3)):
        pool = [it for it in items if it["model"] == mname]
        sample += rng.sample(pool, n)
    rng.shuffle(sample)
    assert len(sample) == 10

    for it in sample:
        rec = run_judge(it)
        print("=" * 70)
        print(f"model={it['model']}  eval_id={it['eval_id']}  parsed={rec['parsed']}")
        print("--- raw judge output ---")
        print(rec["raw_judge_output"])
        if rec["parsed"]:
            print("--- scores ---", {k: rec[k] for k in SCORE_KEYS})
            print("reason:", rec["brief_reason"])
        else:
            print("--- PARSE ERROR ---", rec["parse_error"])
    print("=" * 70)
    print("TEST MODE COMPLETE. Every sample above should show parsed=True.")
    print("If any show parsed=False, fix the prompt/parsing before the full run.")
    print("Then set TEST_MODE = False in this cell, re-run it, and run the next cell.")

## 9. Full run — all 720 judgments, incremental writes, resume-safe

In [ ]:
if TEST_MODE:
    print("TEST_MODE is still True — full run skipped. Flip the flag in the previous cell first.")
else:
    OUT_PATH = "RY3_EVAL_REPAIR_judgments.jsonl"
    done = set()
    if os.path.exists(OUT_PATH):
        with open(OUT_PATH, encoding="utf-8") as f:
            for line in f:
                try:
                    d = json.loads(line)
                    done.add((d["eval_id"], d["model"]))
                except Exception:
                    pass
        print(f"resuming: {len(done)} judgments already on disk")

    fails = 0
    judged = 0
    with open(OUT_PATH, "a", encoding="utf-8") as f:
        for i, it in enumerate(items, 1):
            if (it["eval_id"], it["model"]) in done:
                continue
            rec = run_judge(it)
            judged += 1
            if not rec["parsed"]:
                fails += 1
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            f.flush()  # write-as-you-go: a crash loses nothing already judged
            if i % 50 == 0:
                print(f"[{i}/720] judged={judged} parse_failures={fails} ({fails/max(judged,1)*100:.1f}%)", flush=True)
    print(f"FULL RUN DONE: {judged} new judgments, {fails} parse failures.")

## 10. Summary — parse rates, mean scores, per-eval-id comparison

In [ ]:
judgments = []
with open("RY3_EVAL_REPAIR_judgments.jsonl", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            judgments.append(json.loads(line))
print(f"total judgments on disk: {len(judgments)}")

by_model = collections.defaultdict(list)
for j in judgments:
    by_model[j["model"]].append(j)

print(f"\n{'model':<6} {'n':>4} {'parsed':>7} {'parse_rate':>10} {'mean_total_/20':>14}")
for mname in ("base", "v01", "v02"):
    rows = by_model.get(mname, [])
    ok = [r for r in rows if r.get("parsed")]
    rate = 100.0 * len(ok) / len(rows) if rows else 0.0
    totals = [sum(r[k] for k in SCORE_KEYS) for r in ok]
    mean = sum(totals) / len(totals) if totals else float("nan")
    print(f"{mname:<6} {len(rows):>4} {len(ok):>7} {rate:>9.1f}% {mean:>14.2f}")

score_of = {}
for j in judgments:
    score_of[(j["eval_id"], j["model"])] = sum(j[k] for k in SCORE_KEYS) if j.get("parsed") else None

print(f"\n{'eval_id':<28} {'base':>6} {'v01':>6} {'v02':>6}")
for eid in sorted({j["eval_id"] for j in judgments}):
    def fmt(v):
        return f"{v:>6}" if v is not None else "  FAIL"
    print(f"{eid:<28}{fmt(score_of.get((eid,'base')))}{fmt(score_of.get((eid,'v01')))}{fmt(score_of.get((eid,'v02')))}")